# Serve @examples/pvlib-python as a live endpoint

Turns this Colab session into the endpoint that PowerAI Hub's **Tool tab** points at,
so you can show a primitive being *called* rather than downloaded.

**Run all cells** (Runtime → Run all), then paste the URL cell 4 prints into the
primitive's Tool tab → *Configure endpoint*.

### Read this before demoing

* **The endpoint dies with this session.** Colab reclaims idle notebooks after ~90
  minutes and caps sessions around 12 hours. The Hub stores one address per
  primitive, so when this session ends that address is dead until you re-run and
  paste a new one. Fine for a live demo, wrong for anything left running.
* **Each person who runs this gets a different URL.** One Hub, one address — so this
  is a demo *you* drive, not something visitors clicking Run in Colab wire up
  themselves.
* **Cell 4 downloads `cloudflared`** (Cloudflare's official release) to open a public
  HTTPS tunnel, because a Colab VM has no inbound networking of its own.
* For anything durable, host the same service properly — Lambda, Fly, or the box
  already serving the Hub API.


In [ ]:
!pip -q install pvlib pandas fastapi "uvicorn[standard]"
print("installed")


In [ ]:
# The endpoint that answers the `clearsky_irradiance` tool interface
# @examples/pvlib-python declares. Same code as
# examples/pvlib-clearsky-endpoint/main.py in the Hub repo.
main_py = r'''
from __future__ import annotations

from datetime import date as Date
from typing import Annotated, Literal

import pandas as pd
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
from pvlib.location import Location
from pydantic import BaseModel, Field, ValidationError

app = FastAPI(title="pvlib clear-sky endpoint", version="0.1.0")

PROTOCOL = "1"
SAMPLE_FREQ = "15min"
HOURS_PER_SAMPLE = 0.25


class ClearskyRequest(BaseModel):
    latitude: Annotated[float, Field(ge=-90, le=90)]
    longitude: Annotated[float, Field(ge=-180, le=180)]
    date: Annotated[Date, Field(description="UTC day, YYYY-MM-DD")]
    # 'haurwitz' returns GHI only, so it is excluded rather than padded with nulls:
    # the declared interface promises DNI and DHI too, and a contract that is true
    # for some inputs is not true.
    model: Literal["ineichen", "simplified_solis"] = "ineichen"


class ClearskyResponse(BaseModel):
    ghi_peak_w_m2: float
    dni_peak_w_m2: float
    dhi_peak_w_m2: float
    ghi_daily_wh_m2: float
    model: str


def clearsky(req: ClearskyRequest) -> ClearskyResponse:
    location = Location(req.latitude, req.longitude, tz="UTC")
    times = pd.date_range(
        start=f"{req.date} 00:00", end=f"{req.date} 23:59", freq=SAMPLE_FREQ, tz="UTC"
    )
    frame = location.get_clearsky(times, model=req.model)
    return ClearskyResponse(
        ghi_peak_w_m2=round(float(frame["ghi"].max()), 1),
        dni_peak_w_m2=round(float(frame["dni"].max()), 1),
        dhi_peak_w_m2=round(float(frame["dhi"].max()), 1),
        ghi_daily_wh_m2=round(float(frame["ghi"].sum()) * HOURS_PER_SAMPLE, 1),
        model=req.model,
    )


def _failure(code: str, message: str, status: int) -> JSONResponse:
    # `ok` is the load-bearing field: an agent platform hands the model the response
    # body as a string and never sees the HTTP status, so without it a well-formed
    # error is indistinguishable from a result.
    return JSONResponse(
        {"powerai": PROTOCOL, "ok": False, "error": {"code": code, "message": message}},
        status_code=status,
    )


@app.post("/clearsky")
async def post_clearsky(request: Request) -> JSONResponse:
    try:
        body = await request.json()
    except Exception:
        return _failure("invalid_input", "Request body is not valid JSON.", 400)
    if not isinstance(body, dict):
        return _failure("invalid_input", "Request body must be a JSON object.", 400)

    # Accept the envelope, and the bare arguments object for older callers.
    arguments = body.get("input") if body.get("powerai") else body
    if not isinstance(arguments, dict):
        return _failure("invalid_input", "`input` must be a JSON object.", 400)

    try:
        parsed = ClearskyRequest.model_validate(arguments)
    except ValidationError as exc:
        first = exc.errors()[0]
        field = ".".join(str(p) for p in first.get("loc", ())) or "input"
        return _failure("invalid_input", f"{field}: {first.get('msg', 'invalid')}", 400)

    try:
        result = clearsky(parsed)
    except Exception:
        return _failure("internal", "The calculation failed.", 500)

    return JSONResponse({"powerai": PROTOCOL, "ok": True, "output": result.model_dump()})


@app.get("/health")
def health() -> dict[str, str]:
    return {"status": "ok"}
'''

open("main.py", "w").write(main_py)

# Physics, not just plumbing: summer must beat winter and the polar night must be
# zero, which is what catches a mixed-up column or a broken Wh conversion.
import importlib, sys
sys.path.insert(0, ".")
mod = importlib.import_module("main")
summer = mod.clearsky(mod.ClearskyRequest(latitude=37.0, longitude=-122.0, date="2026-06-21"))
winter = mod.clearsky(mod.ClearskyRequest(latitude=37.0, longitude=-122.0, date="2026-12-21"))
polar = mod.clearsky(mod.ClearskyRequest(latitude=78.0, longitude=15.0, date="2026-12-21"))
assert summer.ghi_peak_w_m2 > winter.ghi_peak_w_m2 > 0, (summer, winter)
assert polar.ghi_peak_w_m2 == 0.0, polar
print("summer", summer.model_dump())
print("polar ", polar.model_dump())
print("self-check OK")


In [ ]:
import re, subprocess, time, urllib.request

PORT = 8011

# Colab VMs have no inbound networking, so a tunnel is the only way in.
# Cloudflare quick tunnels need no account and no credentials.
subprocess.run(
    "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/"
    "cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared",
    shell=True, check=True,
)

server = subprocess.Popen(
    ["uvicorn", "main:app", "--host", "127.0.0.1", "--port", str(PORT)],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

# Wait for the app itself before exposing it, so the tunnel never fronts a dead port.
for _ in range(60):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=1)
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("uvicorn did not come up")
print(f"local endpoint up on :{PORT}")

tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

PUBLIC_URL = ""
deadline = time.time() + 90
while time.time() < deadline:
    line = tunnel.stdout.readline()
    if not line:
        break
    found = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if found:
        PUBLIC_URL = found.group(0)
        break
if not PUBLIC_URL:
    raise RuntimeError("no tunnel URL; re-run this cell")

ENDPOINT_URL = f"{PUBLIC_URL}/clearsky"
print()
print("Paste this into the Tool tab -> Configure endpoint:")
print("   ", ENDPOINT_URL)


In [ ]:
import json, urllib.request

# Prove the public URL answers the enveloped protocol before you paste it anywhere.
body = json.dumps({
    "powerai": "1",
    "tool": "clearsky_irradiance",
    "input": {"latitude": 37.0, "longitude": -122.0, "date": "2026-06-21"},
}).encode()

request = urllib.request.Request(
    ENDPOINT_URL, data=body, headers={"Content-Type": "application/json"}
)
reply = json.load(urllib.request.urlopen(request, timeout=60))
print(json.dumps(reply, indent=1))
assert reply.get("ok") is True, reply
assert set(reply["output"]) >= {
    "ghi_peak_w_m2", "dni_peak_w_m2", "dhi_peak_w_m2", "ghi_daily_wh_m2"
}, reply["output"]

# And that a bad request comes back as ok:false rather than as something a caller
# could mistake for a result.
bad = json.dumps({"powerai": "1", "tool": "clearsky_irradiance",
                  "input": {"longitude": -122.0, "date": "2026-06-21"}}).encode()
try:
    urllib.request.urlopen(urllib.request.Request(
        ENDPOINT_URL, data=bad, headers={"Content-Type": "application/json"}), timeout=30)
    raise AssertionError("a request missing latitude should not succeed")
except urllib.error.HTTPError as err:
    failure = json.load(err)
    assert failure["ok"] is False and failure["error"]["code"] == "invalid_input", failure
    print("error path ->", failure["error"]["message"])

print()
print("public endpoint verified")


## Now show it being called

1. Open `@examples/pvlib-python` on the Hub → **Tool tab** → *Configure endpoint*, and
   paste the URL from cell 4. The Hub requires public HTTPS, which the tunnel URL is.
2. The primitive now reports `invocable: true` and appears in
   `GET /api/artifacts/?invocable=true`.
3. It also appears in the action registry, rendered as an instruction block for an
   agent platform:

   ```
   curl "https://hub-api.powerai.ai/api/integrations/action-registry/?as=markdown"
   ```

4. Or call it directly, which is the same request the registry describes:

   ```
   curl -X POST <the URL from cell 4> \
     -H 'Content-Type: application/json' \
     -d '{"powerai":"1","tool":"clearsky_irradiance",
          "input":{"latitude":37.0,"longitude":-122.0,"date":"2026-06-21"}}'
   ```

**When you are done**, clear the endpoint so the catalogue is not advertising a dead
address: Tool tab → *Configure endpoint* → remove.
